# Problem 5: Study Quiz Generator

**Objective:** build a web application that turns class notes into an interactive multiple-choice quiz with automatic scoring.

The key technique in this project is asking the AI to respond in **JSON**, a structured format that programs can read reliably, instead of ordinary sentences. Real applications use this technique to turn AI output into interactive features.

**Learning outcomes:** prompt design, structured (JSON) output, and building an interactive application from AI output.

### How to use this notebook
- A notebook is made up of **cells**. To run a cell, select it and press **Shift + Enter** (or select the run button).
- On the first run, run the cells **in order, from top to bottom**.
- Comments marked **STUDENT TODO** show where changes are expected:
  - **Level 1:** text changes. No coding experience required.
  - **Level 2:** changes to settings and values.
  - **Level 3:** new features. Mentors are available to help.
- After changing a cell, **re-run that cell and then re-run the "Launch the application" cell** for the change to take effect.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

### Obtaining a free Groq API key (one per team; about 2 minutes)

Groq is a free service that hosts **open-weight AI models**, such as Meta's Llama. The code in this notebook sends a message to Groq and receives the model's response.

1. Go to **https://console.groq.com** and sign in with a Google or GitHub account.
2. Select **API Keys** → **Create API Key**, enter any name, and **copy** the key (it begins with `gsk_`).
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GROQ_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it. The Secrets panel keeps it private.
>
> **No key yet:** the notebook runs in **demonstration mode** with sample responses, so you can explore and edit it before a key is available.

## Step 0: Installation and setup (run both cells)

In [ ]:
# Installs the required libraries. This takes about 30 seconds.
!pip install -q "gradio>=5,<6" openai

In [ ]:
# SETUP: run this cell once. No changes are required here.
import os
import openai
from openai import OpenAI

def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

GROQ_API_KEY = _get_secret("GROQ_API_KEY")
DEMO_MODE = not GROQ_API_KEY

# Groq uses the same interface as the OpenAI API, so the openai library is used
# and pointed at Groq's servers.
client = None if DEMO_MODE else OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

def _friendly_error(e):
    if isinstance(e, openai.AuthenticationError):
        return "Error: the GROQ_API_KEY was rejected. Check that the complete key (beginning with gsk_) is saved in Colab Secrets, then re-run the setup cell."
    if isinstance(e, openai.RateLimitError):
        return "Error: the Groq free-tier usage limit has been reached. Wait about one minute and try again, or set MODEL = 'llama-3.1-8b-instant'."
    if isinstance(e, openai.NotFoundError):
        return f"Error: Groq does not recognise the model '{MODEL}'. Check the spelling, or see https://console.groq.com/docs/models"
    if isinstance(e, openai.APIConnectionError):
        return "Error: unable to connect to Groq. Check the internet connection and try again."
    return f"Error: the request to the AI service failed ({e})."

def ask_llm(messages, temperature=0.7, json_mode=False, demo_reply="(Demonstration mode response)"):
    """Sends a list of chat messages to the AI model and returns its response as text.
    In demonstration mode, returns `demo_reply` instead. If the request fails,
    returns a message beginning with "Error:" instead of stopping the program."""
    if DEMO_MODE:
        return demo_reply
    try:
        extra = {"response_format": {"type": "json_object"}} if json_mode else {}
        response = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature, **extra
        )
        return response.choices[0].message.content
    except Exception as e:
        return _friendly_error(e)

if DEMO_MODE:
    print("Demonstration mode: no GROQ_API_KEY was found, so the application will show sample responses.")
    print("To connect the AI model: Secrets (left sidebar) → add GROQ_API_KEY → enable Notebook access → re-run this cell.")
else:
    print("Connected to Groq. The AI model is ready.")

## Step 1: Configure the quiz generator

`QUIZ_INSTRUCTIONS` is the **prompt** that tells the AI how to write the questions. The values `{num_questions}` and `{difficulty}` in curly braces are filled in automatically.

**Important:** do not change the **JSON format** section of the prompt (unless attempting a Level 3 extension). The application depends on that exact structure.

In [ ]:
# STUDENT TODO (Level 1): set the application name and description.
APP_TITLE = "Study Quiz Generator"
APP_DESCRIPTION = "Paste your notes to generate a practice quiz. Powered by open-weight Llama."

# STUDENT TODO (Level 2): the instructions for the AI. Adjust the style or focus of the questions,
#   but keep the JSON format section at the end.
QUIZ_INSTRUCTIONS = """
You are a teacher who writes multiple-choice quizzes for college students.
Write exactly {num_questions} questions based ONLY on the notes the student provides.
Difficulty: {difficulty}.
- easy: test basic facts and definitions from the notes.
- medium: test understanding. Ask "why" and "how", not only "what".
- hard: test the application of ideas to new situations, with plausible incorrect options.
Each question must have exactly 4 options and exactly one correct answer.

Reply with JSON only, in exactly this format:
{{
  "questions": [
    {{
      "question": "the question text",
      "options": ["option A", "option B", "option C", "option D"],
      "answer_index": 0,
      "explanation": "one sentence explaining why the answer is correct"
    }}
  ]
}}
answer_index is the position of the correct option: 0, 1, 2 or 3.
"""

# STUDENT TODO (Level 2): default settings.
DEFAULT_NUM_QUESTIONS = 5
MAX_QUESTIONS = 10            # the maximum number of questions the application can show
DIFFICULTIES = ["easy", "medium", "hard"]
TEMPERATURE = 0.4             # lower values give more accurate questions; higher values give more variety

# STUDENT TODO (Level 2): select the open-weight model that generates responses.
#   "llama-3.1-8b-instant"     Meta Llama 3.1, 8 billion parameters. Fast, highest free limits. (default)
#   "llama-3.3-70b-versatile"  Meta Llama 3.3, 70 billion parameters. More capable, slower, lower daily limit.
#   "openai/gpt-oss-20b"       OpenAI's open-weight gpt-oss model.
#   Full list: https://console.groq.com/docs/models
MODEL = "llama-3.1-8b-instant"

# STUDENT TODO (Level 1): replace with notes from one of your own courses.
SAMPLE_NOTES = """
Photosynthesis is the process plants use to turn light energy into chemical energy.
It happens mainly in the leaves, inside tiny structures called chloroplasts, which contain
a green pigment called chlorophyll. Plants take in carbon dioxide from the air through small
pores called stomata, and water from the soil through their roots. Using sunlight, they convert
these into glucose (a sugar used for energy and growth) and release oxygen as a by-product.
The overall equation is: 6CO2 + 6H2O + light energy -> C6H12O6 + 6O2.
Photosynthesis has two stages: the light-dependent reactions, which happen in the thylakoid
membranes and produce ATP and NADPH, and the Calvin cycle, which happens in the stroma and
uses that ATP and NADPH to build glucose.
"""

print("Quiz settings configured.")

## Step 2: The quiz generation function (run this cell)

`generate_quiz` sends the notes to the AI and converts the JSON response into a list of questions. It also **validates the response**, because AI models occasionally produce output in the wrong format.

In [ ]:
import json

# A sample quiz used in demonstration mode (when no API key is set).
DEMO_QUIZ = {"questions": [
    {"question": "(Sample) Where in the plant cell does photosynthesis mainly take place?",
     "options": ["Mitochondria", "Chloroplasts", "Nucleus", "Cell wall"], "answer_index": 1,
     "explanation": "Chloroplasts contain chlorophyll, which captures light energy."},
    {"question": "(Sample) Which gas do plants release during photosynthesis?",
     "options": ["Carbon dioxide", "Nitrogen", "Oxygen", "Hydrogen"], "answer_index": 2,
     "explanation": "Oxygen is released as a by-product when water is split."},
    {"question": "(Sample) Through which structures do plants take in carbon dioxide?",
     "options": ["Stomata", "Roots", "Xylem", "Petals"], "answer_index": 0,
     "explanation": "Stomata are small pores, mostly on the underside of leaves."},
]}


def generate_quiz(notes, difficulty="medium", num_questions=DEFAULT_NUM_QUESTIONS):
    """Returns a list of questions, each with "question", "options", "answer_index" and "explanation"."""
    messages = [
        {"role": "system", "content": QUIZ_INSTRUCTIONS.format(num_questions=num_questions, difficulty=difficulty)},
        {"role": "user", "content": f"Here are my notes:\n\n{notes}"},
    ]
    reply = ask_llm(messages, temperature=TEMPERATURE, json_mode=True, demo_reply=json.dumps(DEMO_QUIZ))
    if reply.startswith("Error:"):
        raise ValueError(reply)

    data = json.loads(reply)

    # Validate the response: keep only questions with the correct structure.
    questions = []
    for q in data.get("questions", []):
        options = [str(o) for o in q.get("options", [])]
        answer = q.get("answer_index")
        if q.get("question") and len(options) >= 2 and isinstance(answer, int) and 0 <= answer < len(options):
            questions.append({
                "question": str(q["question"]),
                "options": options,
                "answer_index": answer,
                "explanation": str(q.get("explanation", "")),
            })

    # STUDENT TODO (Level 3): randomise the order of the questions or options here.
    #   Hint: import random; random.shuffle(questions)
    #   (If the options are shuffled, answer_index must be updated as well.)
    return questions[:MAX_QUESTIONS]


# Quick test: generate a quiz from the sample notes and print it.
for i, q in enumerate(generate_quiz(SAMPLE_NOTES, "easy", 3), start=1):
    print(f"Q{i}. {q['question']}")
    for j, option in enumerate(q["options"]):
        print(f"   {'[correct]' if j == q['answer_index'] else '         '} {option}")
    print(f"   Explanation: {q['explanation']}\n")

## Step 3: Launch the application

Run this cell. The application appears below, and a **public link** is also created for sharing. Re-run this cell after changing any cell above.

In [ ]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running


def on_generate(notes, difficulty, num_questions):
    """Runs when "Generate quiz" is selected. Fills in the question fields."""
    if not notes.strip():
        raise gr.Error("Paste some notes first.")
    try:
        quiz = generate_quiz(notes, difficulty, int(num_questions))
    except json.JSONDecodeError:
        raise gr.Error("The AI response was not valid JSON. Please try again.")
    except ValueError as e:
        raise gr.Error(str(e))
    if not quiz:
        raise gr.Error("The AI did not return any usable questions. Try again or provide more notes.")

    status = f"### Quiz ({difficulty}): {len(quiz)} questions"
    updates = []
    for i in range(MAX_QUESTIONS):
        if i < len(quiz):
            updates.append(gr.update(visible=True, label=f"Q{i + 1}. {quiz[i]['question']}",
                                     choices=quiz[i]["options"], value=None))
        else:
            updates.append(gr.update(visible=False, value=None))
    return [quiz, status, ""] + updates


def on_check(quiz, *answers):
    """Runs when "Check answers" is selected. Scores the quiz."""
    if not quiz:
        return "Generate a quiz first."
    score, feedback = 0, []
    for i, q in enumerate(quiz):
        correct = q["options"][q["answer_index"]]
        if answers[i] == correct:
            score += 1
            feedback.append(f"**Q{i + 1}: Correct.** {q['explanation']}")
        else:
            picked = answers[i] or "no answer"
            feedback.append(f"**Q{i + 1}: Incorrect.** Your answer: *{picked}*. Correct answer: **{correct}**. {q['explanation']}")

    # STUDENT TODO (Level 2): change the feedback message for each score range.
    percent = score / len(quiz)
    if percent == 1:
        summary = "All answers are correct."
    elif percent >= 0.6:
        summary = "Good result. Review the questions answered incorrectly."
    else:
        summary = "Review your notes and try a new quiz."
    return f"## Score: {score}/{len(quiz)}\n{summary}\n\n" + "\n\n".join(feedback)


with gr.Blocks(theme=gr.themes.Soft(), title=APP_TITLE) as demo:  # STUDENT TODO (Level 2): alternative theme: gr.themes.Glass()
    quiz_state = gr.State([])  # stores the current quiz

    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    notes_box = gr.Textbox(value=SAMPLE_NOTES.strip(), lines=8, label="Notes")
    with gr.Row():
        difficulty_box = gr.Radio(DIFFICULTIES, value="medium", label="Difficulty")
        num_box = gr.Slider(1, MAX_QUESTIONS, value=DEFAULT_NUM_QUESTIONS, step=1, label="Number of questions")
    generate_btn = gr.Button("Generate quiz", variant="primary")

    status_md = gr.Markdown()
    question_boxes = [gr.Radio(choices=[], visible=False, interactive=True) for _ in range(MAX_QUESTIONS)]
    check_btn = gr.Button("Check answers")
    result_md = gr.Markdown()

    generate_btn.click(on_generate, inputs=[notes_box, difficulty_box, num_box],
                       outputs=[quiz_state, status_md, result_md] + question_boxes)
    check_btn.click(on_check, inputs=[quiz_state] + question_boxes, outputs=result_md)

demo.launch(share=True)

## Level 3: Optional extensions

**1. Testing with real course material (recommended for the demonstration).** Paste notes from a current course and check the questions for accuracy. Record how many out of 10 questions are correct and well written, at each difficulty level.

**2. Topic-only mode.** Allow users to enter only a topic (for example, "the French Revolution") instead of notes. Consider what must change in the prompt, and the associated risk: without source notes, the AI may produce inaccurate information.

**3. Additional question types.** Add true/false questions by changing the prompt so that some questions use `["True", "False"]` as options. The application already supports any number of options.

**4. Hints.** Ask the AI to include a `"hint"` field for each question in the JSON, and show the hints through an additional button.

**5. Answer verification.** After generating the quiz, call `ask_llm` a second time: "Here are some quiz questions and the source notes. Is each answer supported by the notes? Reply in JSON." Remove any questions that fail verification.

**6. Flashcard export.** Add a button that converts the quiz into a list of "Q: ... / A: ..." flashcards in a text field that can be copied.

## Demonstration checklist
- [ ] A quiz generated live from real course notes
- [ ] Difficulty levels that produce clearly different questions
- [ ] The team can explain why the AI is asked to respond in JSON
- [ ] One example of an inaccuracy produced by the AI, and how the team addressed it

## Troubleshooting

| Issue | Resolution |
|---|---|
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The application does not reflect recent changes | Re-run the edited cell **and** the "Launch the application" cell. |
| The notebook remains in demonstration mode | Add `GROQ_API_KEY` in Secrets, enable **Notebook access**, and re-run the setup cell. |
| "Usage limit reached" error | Wait about one minute. Free keys allow a limited number of requests per minute. |
| `SyntaxError` | Usually caused by a missing quotation mark `"`, bracket `)` or comma. Check the line shown in the error. |
| Unexpected behaviour | Select **Runtime → Restart session**, then run all cells from the top. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |